# Conversão e tratamento dos dados

### Conversão da base do Economatica de CSV para Parquet

Nesta etapa, o objetivo é carregar o arquivo bruto extraído do Economatica em formato CSV e convertê-lo para o formato Parquet.

O formato Parquet é mais eficiente para bases grandes, pois ocupa menos espaço, carrega mais rápido e facilita o trabalho nas próximas etapas do projeto.

In [1]:
import pandas as pd
from pathlib import Path

In [4]:
arquivo_entrada = Path("../dados/dados_economatica_B3.csv")
arquivo_saida = Path("../dados_tratados/dados_economatica_B3.parquet")

In [15]:
df = pd.read_csv(
    arquivo_entrada,
    sep=",",
    decimal=".",
    encoding="latin1"
)
df.head()

,Ativo,Data,Fechamento|ajust p/ prov|Em moeda orig,Abertura|ajust p/ prov|Em moeda orig,Mínimo|ajust p/ prov|Em moeda orig,Máximo|ajust p/ prov|Em moeda orig,Médio|ajust p/ prov|Em moeda orig,Q Negs,Volume$|Em moeda orig|em milhares,Q Títs|ajust p/ prov|em milhares
0,TTEN3<XBSP>,2010-01-01,-,-,-,-,-,-,-,-
1,TTEN3<XBSP>,2010-01-04,-,-,-,-,-,-,-,-
2,TTEN3<XBSP>,2010-01-05,-,-,-,-,-,-,-,-
3,TTEN3<XBSP>,2010-01-06,-,-,-,-,-,-,-,-
4,TTEN3<XBSP>,2010-01-07,-,-,-,-,-,-,-,-


In [16]:
df.shape

(6053454, 10)

In [17]:
df.columns


Index(['Ativo', 'Data', 'Fechamento|ajust p/ prov|Em moeda orig',
       'Abertura|ajust p/ prov|Em moeda orig',
       'Mínimo|ajust p/ prov|Em moeda orig',
       'Máximo|ajust p/ prov|Em moeda orig',
       'Médio|ajust p/ prov|Em moeda orig', 'Q Negs',
       'Volume$|Em moeda orig|em milhares',
       'Q Títs|ajust p/ prov|em milhares'],
      dtype='str')

In [18]:
df_parquet = df.copy()
df_parquet.columns = (
    df_parquet.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(".", "", regex=False)
)

In [19]:
df_parquet.columns

Index(['ativo', 'data', 'fechamento|ajust_p__prov|em_moeda_orig',
       'abertura|ajust_p__prov|em_moeda_orig',
       'mínimo|ajust_p__prov|em_moeda_orig',
       'máximo|ajust_p__prov|em_moeda_orig',
       'médio|ajust_p__prov|em_moeda_orig', 'q_negs',
       'volume$|em_moeda_orig|em_milhares',
       'q_títs|ajust_p__prov|em_milhares'],
      dtype='str')

In [20]:
df_parquet.to_parquet(arquivo_saida, index=False)

In [21]:
df_teste = pd.read_parquet(arquivo_saida)
df_teste.head()

,ativo,data,fechamento|ajust_p__prov|em_moeda_orig,abertura|ajust_p__prov|em_moeda_orig,mínimo|ajust_p__prov|em_moeda_orig,máximo|ajust_p__prov|em_moeda_orig,médio|ajust_p__prov|em_moeda_orig,q_negs,volume$|em_moeda_orig|em_milhares,q_títs|ajust_p__prov|em_milhares
0,TTEN3<XBSP>,2010-01-01,-,-,-,-,-,-,-,-
1,TTEN3<XBSP>,2010-01-04,-,-,-,-,-,-,-,-
2,TTEN3<XBSP>,2010-01-05,-,-,-,-,-,-,-,-
3,TTEN3<XBSP>,2010-01-06,-,-,-,-,-,-,-,-
4,TTEN3<XBSP>,2010-01-07,-,-,-,-,-,-,-,-


In [22]:
print("Base original:", df.shape)
print("Base Parquet:", df_teste.shape)

Base original: (6053454, 10)
Base Parquet: (6053454, 10)


### Tratamento da base do Economatica

O objetivo é tratar a base de cotações extraída do Economatica e convertida para Parquet. A base contém dados diários de ações da B3. 

Cuidados metodológicos desta etapa:

- Não remover ativos apenas porque deixaram de ser negociados.
- Não selecionar ativos com base no período completo.
- Não preencher preços faltantes com preços anteriores.
- Não usar dados futuros.
- Não eliminar empresas por não existirem no último pregão.
- Remover apenas observações sem preço válido, pois elas não podem ser usadas em cálculos de retorno, cointegração ou spread.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import unicodedata

In [3]:
arquivo_entrada = Path("../dados_tratados/dados_economatica_B3.parquet")

df = pd.read_parquet(arquivo_entrada)

print("Tamanho da base bruta:", df.shape)
display(df.head())

Tamanho da base bruta: (6053454, 10)


,ativo,data,fechamento|ajust_p__prov|em_moeda_orig,abertura|ajust_p__prov|em_moeda_orig,mínimo|ajust_p__prov|em_moeda_orig,máximo|ajust_p__prov|em_moeda_orig,médio|ajust_p__prov|em_moeda_orig,q_negs,volume$|em_moeda_orig|em_milhares,q_títs|ajust_p__prov|em_milhares
0,TTEN3<XBSP>,2010-01-01,-,-,-,-,-,-,-,-
1,TTEN3<XBSP>,2010-01-04,-,-,-,-,-,-,-,-
2,TTEN3<XBSP>,2010-01-05,-,-,-,-,-,-,-,-
3,TTEN3<XBSP>,2010-01-06,-,-,-,-,-,-,-,-
4,TTEN3<XBSP>,2010-01-07,-,-,-,-,-,-,-,-


In [4]:
df.columns

Index(['ativo', 'data', 'fechamento|ajust_p__prov|em_moeda_orig',
       'abertura|ajust_p__prov|em_moeda_orig',
       'mínimo|ajust_p__prov|em_moeda_orig',
       'máximo|ajust_p__prov|em_moeda_orig',
       'médio|ajust_p__prov|em_moeda_orig', 'q_negs',
       'volume$|em_moeda_orig|em_milhares',
       'q_títs|ajust_p__prov|em_milhares'],
      dtype='str')

### Padronização dos nomes das colunas

As colunas originais do Economatica possuem acentos, espaços e símbolos especiais. Para facilitar o tratamento, os nomes serão padronizados.

In [5]:
def normalizar_nome_coluna(nome):
    
    nome = str(nome)
    
    nome = nome.strip()
    
    nome = nome.lower()
    
    nome = unicodedata.normalize("NFKD", nome)
    nome = "".join([c for c in nome if not unicodedata.combining(c)])
    
    nome = nome.replace(" ", "_")
    nome = nome.replace("/", "_")
    nome = nome.replace("$", "")
    nome = nome.replace("|", "_")
    
    nome = nome.replace("(", "")
    nome = nome.replace(")", "")
    nome = nome.replace('"', "")
    nome = nome.replace("'", "")
    
    while "__" in nome:
        nome = nome.replace("__", "_")
    
    nome = nome.strip("_")
    
    return nome


df.columns = [normalizar_nome_coluna(col) for col in df.columns]

df.columns

Index(['ativo', 'data', 'fechamento_ajust_p_prov_em_moeda_orig',
       'abertura_ajust_p_prov_em_moeda_orig',
       'minimo_ajust_p_prov_em_moeda_orig',
       'maximo_ajust_p_prov_em_moeda_orig', 'medio_ajust_p_prov_em_moeda_orig',
       'q_negs', 'volume_em_moeda_orig_em_milhares',
       'q_tits_ajust_p_prov_em_milhares'],
      dtype='str')

### Renomeação das colunas

Depois da padronização, as colunas serão renomeadas para nomes mais simples e claros.

A base final terá colunas como ticker, data, fechamento ajustado, abertura ajustada, volume financeiro e quantidade de negócios.

In [6]:
df = df.rename(columns={
    "ativo": "ativo",
    "data": "data",
    "fechamento_ajust_p_prov_em_moeda_orig": "fechamento_ajustado",
    "abertura_ajust_p_prov_em_moeda_orig": "abertura_ajustada",
    "minimo_ajust_p_prov_em_moeda_orig": "minimo_ajustado",
    "maximo_ajust_p_prov_em_moeda_orig": "maximo_ajustado",
    "medio_ajust_p_prov_em_moeda_orig": "medio_ajustado",
    "q_negs": "q_negs",
    "volume_em_moeda_orig_em_milhares": "volume_milhares",
    "q_tits_ajust_p_prov_em_milhares": "q_titulos_milhares"
})

df.columns

Index(['ativo', 'data', 'fechamento_ajustado', 'abertura_ajustada',
       'minimo_ajustado', 'maximo_ajustado', 'medio_ajustado', 'q_negs',
       'volume_milhares', 'q_titulos_milhares'],
      dtype='str')

In [7]:
colunas_necessarias = [
    "ativo",
    "data",
    "fechamento_ajustado",
    "abertura_ajustada",
    "minimo_ajustado",
    "maximo_ajustado",
    "medio_ajustado",
    "q_negs",
    "volume_milhares",
    "q_titulos_milhares"
]

colunas_faltantes = [col for col in colunas_necessarias if col not in df.columns]

if colunas_faltantes:
    raise ValueError(f"Colunas faltantes: {colunas_faltantes}")

print("Todas as colunas necessárias estão presentes.")

Todas as colunas necessárias estão presentes.


### Conversão da coluna de data

A coluna de data será convertida para o formato datetime do pandas.

In [8]:
df["data"] = pd.to_datetime(df["data"], errors="coerce")

print(df["data"].min())
print(df["data"].max())

2010-01-01 00:00:00
2026-05-08 00:00:00


### Padronização dos códigos dos ativos

A coluna original de ativo será preservada, mas será criada uma coluna adicional chamada ticker.

O ticker remove o sufixo da bolsa, como <..>, mantendo apenas o código negociado do ativo.

In [11]:
df["ativo"] = df["ativo"].astype(str).str.strip().str.upper()

df["ticker"] = (
    df["ativo"]
    .str.replace(r"<.*?>", "", regex=True)
    .str.strip()
)

df[["ativo", "ticker"]].head()

,ativo,ticker
0,TTEN3<XBSP>,TTEN3
1,TTEN3<XBSP>,TTEN3
2,TTEN3<XBSP>,TTEN3
3,TTEN3<XBSP>,TTEN3
4,TTEN3<XBSP>,TTEN3


In [12]:
df["ticker"].unique()[:30]

<ArrowStringArray>
[ 'TTEN3', 'QVUM3B',  'QVQP3',  'APPA3',  'APPA4',  'ABCB3',  'ABCB4',
 'ABCB11',  'ABYA3',  'EALT3',  'EALT4',  'AVIL3',  'AVIL4',  'ADHM3',
  'AERI3',  'AESB3',  'TIET3',  'TIET4',  'AELP3',  'AESL3',  'AESL4',
  'GETI3',  'GETI4',  'AESO3',  'AETA3',  'AFLU3',  'AFLU5',  'AFLU6',
  'AFLT3', 'ANDG3B']
Length: 30, dtype: str

### Conversão das variáveis numéricas

As colunas de preços, volume, quantidade de negócios e quantidade de títulos serão convertidas para formato numérico.

Os valores ausentes representados por "-" serão convertidos para NaN.

Não será feito preenchimento de preços faltantes, pois isso poderia criar preços artificiais e distorcer retornos, cointegração e sinais da estratégia.

In [13]:
colunas_numericas = [
    "fechamento_ajustado",
    "abertura_ajustada",
    "minimo_ajustado",
    "maximo_ajustado",
    "medio_ajustado",
    "q_negs",
    "volume_milhares",
    "q_titulos_milhares"
]

for col in colunas_numericas:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .replace("-", np.nan)
        .replace("", np.nan)
    )
    
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[colunas_numericas].dtypes

fechamento_ajustado    float64
abertura_ajustada      float64
minimo_ajustado        float64
maximo_ajustado        float64
medio_ajustado         float64
q_negs                 float64
volume_milhares        float64
q_titulos_milhares     float64
dtype: object

### Criação do volume financeiro em reais

O volume financeiro veio do Economatica em milhares de reais.

Para facilitar a interpretação e o filtro de liquidez, será criada uma coluna em reais, multiplicando o volume por 1.000.

In [14]:
df["volume_financeiro"] = df["volume_milhares"] * 1000

df["q_titulos"] = df["q_titulos_milhares"] * 1000

df[["volume_milhares", "volume_financeiro", "q_titulos_milhares", "q_titulos"]].head()

,volume_milhares,volume_financeiro,q_titulos_milhares,q_titulos
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN


In [15]:
df[["volume_milhares", "volume_financeiro", "q_titulos_milhares", "q_titulos"]].describe()

,volume_milhares,volume_financeiro,q_titulos_milhares,q_titulos
count,1.374421e+06,1.374421e+06,1.374421e+06,1.374421e+06
mean,3.918437e+04,3.918437e+07,2.519234e+03,2.519234e+06
std,1.478588e+05,1.478588e+08,2.201382e+04,2.201382e+07
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,6.278000e+01,6.278000e+04,5.600000e+00,5.600000e+03
50%,1.624640e+03,1.624640e+06,1.537000e+02,1.537000e+05
75%,2.133929e+04,2.133929e+07,1.593000e+03,1.593000e+06
max,1.152877e+07,1.152877e+10,8.916121e+06,8.916121e+09


In [16]:
df[["volume_milhares", "volume_financeiro", "q_titulos_milhares", "q_titulos"]].notna().sum()

volume_milhares       1374421
volume_financeiro     1374421
q_titulos_milhares    1374421
q_titulos             1374421
dtype: int64

### Remoção de observações sem preço válido

Nessa etapa, serão removidas observações diárias sem data, sem ticker ou sem preço de fechamento ajustado válido.

Isso não gera viés de sobrevivência, pois ativos que deixaram de existir continuam na base nos períodos em que possuíam dados válidos.

In [17]:
linhas_antes = len(df)

df = df.dropna(subset=["data", "ticker"])

df = df[df["ticker"] != ""]

df = df[df["fechamento_ajustado"].notna()]

df = df[df["fechamento_ajustado"] > 0]

linhas_depois = len(df)

print("Linhas removidas:", linhas_antes - linhas_depois)
print("Tamanho após filtro:", df.shape)
print("Quantidade de tickers restantes:", df["ticker"].nunique())

Linhas removidas: 4679055
Tamanho após filtro: (1374399, 13)
Quantidade de tickers restantes: 775


### Observação metodológica sobre dados faltantes

Não foi feito preenchimento de preços faltantes com o preço anterior, pois preencher preços ausentes pode criar retornos artificiais, distorcer a cointegração e gerar sinais falsos.

Por isso, cada ativo será usado apenas nos dias em que tiver preço observado.

### Tratamento de volume e quantidade de negócios

Volume financeiro e quantidade de negócios serão usados posteriormente para selecionar ativos líquidos.

Nesta etapa, não será feita seleção de liquidez. Também não serão removidos ativos por baixa liquidez.

Apenas valores negativos serão removidos, pois não fazem sentido econômico.

In [18]:
linhas_antes = len(df)

df = df[
    (df["volume_financeiro"].isna() | (df["volume_financeiro"] >= 0)) &
    (df["q_negs"].isna() | (df["q_negs"] >= 0)) &
    (df["q_titulos"].isna() | (df["q_titulos"] >= 0))
]

linhas_depois = len(df)

print("Linhas removidas por volume/negócios negativos:", linhas_antes - linhas_depois)
print("Tamanho atual:", df.shape)

Linhas removidas por volume/negócios negativos: 0
Tamanho atual: (1374399, 13)


### Filtro do período de análise

A análise será feita a partir de 2010, conforme definido no projeto.

In [19]:
df = df[df["data"] >= pd.Timestamp("2010-01-01")]

print("Data inicial:", df["data"].min())
print("Data final:", df["data"].max())
print("Tamanho:", df.shape)

Data inicial: 2010-01-04 00:00:00
Data final: 2026-05-08 00:00:00
Tamanho: (1374399, 13)


### Remoção de duplicatas

Para cada ticker e data, deve existir apenas uma observação.

Caso existam duplicatas, será mantida a última ocorrência.

In [20]:
linhas_antes = len(df)

df = df.drop_duplicates(subset=["ticker", "data"], keep="last")

linhas_depois = len(df)

print("Duplicatas removidas:", linhas_antes - linhas_depois)
print("Tamanho atual:", df.shape)

Duplicatas removidas: 0
Tamanho atual: (1374399, 13)


In [21]:
df = df.sort_values(["ticker", "data"]).reset_index(drop=True)

df.head()

,ativo,data,fechamento_ajustado,abertura_ajustada,minimo_ajustado,maximo_ajustado,medio_ajustado,q_negs,volume_milhares,q_titulos_milhares,ticker,volume_financeiro,q_titulos
0,AALR3<XBSP>,2016-10-27,19.723993,19.723993,19.723993,19.723993,19.723993,0.0,0.000,0.0,AALR3,0.0,0.0
1,AALR3<XBSP>,2016-10-28,18.935033,19.033653,18.658897,19.487305,19.023791,4460.0,122334.647,6342.6,AALR3,122334647.0,6342600.0
2,AALR3<XBSP>,2016-10-31,17.810766,18.925171,17.268356,18.935033,17.919248,4238.0,45857.231,2523.3,AALR3,45857231.0,2523300.0
3,AALR3<XBSP>,2016-11-01,17.652974,17.810766,16.923186,18.126350,17.495182,2072.0,17676.981,996.2,AALR3,17676981.0,996200.0
4,AALR3<XBSP>,2016-11-03,17.741732,17.751594,17.071116,17.988282,17.682560,2157.0,11132.994,621.0,AALR3,11132994.0,621000.0


### Seleção das colunas finais

Nesta etapa, selecionamos as colunas que serão usadas nas próximas fases do projeto.

A base final mantém todos os ativos com observações válidas.

In [25]:
colunas_finais = [
    "ticker",
    "ativo",
    "data",
    "fechamento_ajustado",
    "abertura_ajustada",
    "minimo_ajustado",
    "maximo_ajustado",
    "medio_ajustado",
    "q_negs",
    "volume_financeiro",
    "q_titulos"
]

df_tratado = df[colunas_finais].copy()

df_tratado.head()

,ticker,ativo,data,fechamento_ajustado,abertura_ajustada,minimo_ajustado,maximo_ajustado,medio_ajustado,q_negs,volume_financeiro,q_titulos
0,AALR3,AALR3<XBSP>,2016-10-27,19.723993,19.723993,19.723993,19.723993,19.723993,0.0,0.0,0.0
1,AALR3,AALR3<XBSP>,2016-10-28,18.935033,19.033653,18.658897,19.487305,19.023791,4460.0,122334647.0,6342600.0
2,AALR3,AALR3<XBSP>,2016-10-31,17.810766,18.925171,17.268356,18.935033,17.919248,4238.0,45857231.0,2523300.0
3,AALR3,AALR3<XBSP>,2016-11-01,17.652974,17.810766,16.923186,18.126350,17.495182,2072.0,17676981.0,996200.0
4,AALR3,AALR3<XBSP>,2016-11-03,17.741732,17.751594,17.071116,17.988282,17.682560,2157.0,11132994.0,621000.0


### Definição do índice da base tratada

Nesta etapa, definimos um índice duplo com ticker e data.

Como a base possui várias ações negociadas na mesma data, usar apenas a data como índice geraria datas repetidas. Por isso, o índice mais adequado é composto por ticker e data.

Esse formato identifica cada observação de forma única e facilita as próximas etapas do projeto.

In [26]:
df_tratado = df_tratado.set_index(["ticker", "data"]).sort_index()

df_tratado.head()

ativo  fechamento_ajustado  abertura_ajustada  \
ticker data                                                              
AALR3  2016-10-27  AALR3<XBSP>            19.723993          19.723993   
       2016-10-28  AALR3<XBSP>            18.935033          19.033653   
       2016-10-31  AALR3<XBSP>            17.810766          18.925171   
       2016-11-01  AALR3<XBSP>            17.652974          17.810766   
       2016-11-03  AALR3<XBSP>            17.741732          17.751594   

                   minimo_ajustado  maximo_ajustado  medio_ajustado  q_negs  \
ticker data                                                                   
AALR3  2016-10-27        19.723993        19.723993       19.723993     0.0   
       2016-10-28        18.658897        19.487305       19.023791  4460.0   
       2016-10-31        17.268356        18.935033       17.919248  4238.0   
       2016-11-01        16.923186        18.126350       17.495182  2072.0   
       2016-11-03        17.071116        17.988282       17.682560  2157.0   

                   volume_financeiro  q_titulos  
ticker data                                      
AALR3  2016-10-27                0.0        0.0  
       2016-10-28        122334647.0  6342600.0  
       2016-10-31         45857231.0  2523300.0  
       2016-11-01         17676981.0   996200.0  
       2016-11-03         11132994.0   621000.0

### Checagem final da base tratada

Antes de salvar a base, verificamos o tamanho final, o número de tickers, o intervalo de datas e a quantidade de dados ausentes nas principais colunas.

In [27]:
print("Tamanho final:", df_tratado.shape)
print("Quantidade de tickers:", df_tratado.index.get_level_values("ticker").nunique())
print("Data inicial:", df_tratado.index.get_level_values("data").min())
print("Data final:", df_tratado.index.get_level_values("data").max())

df_tratado[[
    "fechamento_ajustado",
    "q_negs",
    "volume_financeiro"
]].isna().sum()

Tamanho final: (1374399, 9)
Quantidade de tickers: 775
Data inicial: 2010-01-04 00:00:00
Data final: 2026-05-08 00:00:00


fechamento_ajustado    0
q_negs                 0
volume_financeiro      0
dtype: int64

In [24]:
ativos_por_ano = (
    df_tratado
    .assign(ano=df_tratado["data"].dt.year)
    .groupby("ano")["ticker"]
    .nunique()
    .reset_index(name="quantidade_tickers")
)

ativos_por_ano

,ano,quantidade_tickers
0,2010,554
1,2011,540
2,2012,514
3,2013,496
4,2014,478
5,2015,477
6,2016,466
7,2017,467
8,2018,460
9,2019,461


### Salvamento da base tratada

A base tratada será salva em formato Parquet.

Esse arquivo será usado na próxima etapa, em que os ativos líquidos serão selecionados dinamicamente dentro de cada janela temporal.

In [28]:
arquivo_saida = Path("../dados_tratados/dados_economatica_B3_tratado.parquet")

arquivo_saida.parent.mkdir(parents=True, exist_ok=True)

df_tratado.to_parquet(arquivo_saida)

print("Arquivo salvo em:", arquivo_saida)

Arquivo salvo em: ..\dados_tratados\dados_economatica_B3_tratado.parquet


In [29]:
df_teste = pd.read_parquet(arquivo_saida)

print("Base salva:", df_teste.shape)
display(df_teste.head())

Base salva: (1374399, 9)


ativo  fechamento_ajustado  abertura_ajustada  \
ticker data                                                              
AALR3  2016-10-27  AALR3<XBSP>            19.723993          19.723993   
       2016-10-28  AALR3<XBSP>            18.935033          19.033653   
       2016-10-31  AALR3<XBSP>            17.810766          18.925171   
       2016-11-01  AALR3<XBSP>            17.652974          17.810766   
       2016-11-03  AALR3<XBSP>            17.741732          17.751594   

                   minimo_ajustado  maximo_ajustado  medio_ajustado  q_negs  \
ticker data                                                                   
AALR3  2016-10-27        19.723993        19.723993       19.723993     0.0   
       2016-10-28        18.658897        19.487305       19.023791  4460.0   
       2016-10-31        17.268356        18.935033       17.919248  4238.0   
       2016-11-01        16.923186        18.126350       17.495182  2072.0   
       2016-11-03        17.071116        17.988282       17.682560  2157.0   

                   volume_financeiro  q_titulos  
ticker data                                      
AALR3  2016-10-27                0.0        0.0  
       2016-10-28        122334647.0  6342600.0  
       2016-10-31         45857231.0  2523300.0  
       2016-11-01         17676981.0   996200.0  
       2016-11-03         11132994.0   621000.0